In [ ]:
!pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu116

In [ ]:
!git clone https://github.com/ultralytics/yolov5

In [ ]:
!cd yolov5 && pip install -r requirements.txt

In [ ]:
import torch
from matplotlib import pyplot as plt
import numpy as np
import cv2
import os

In [ ]:
#Pre-trained model on COCO dataset
model = torch.hub.load('ultralytics/yolov5', 'yolov5s')

In [ ]:
model

In [ ]:
img = 'https://images.hindustantimes.com/auto/img/2024/03/08/600x338/Pakistan_Rowdy_Riders_1709869745622_1709869872815.jpeg'

In [ ]:
result = model(img)
result.print()

In [ ]:
%matplotlib inline
plt.imshow(np.squeeze(result.render()))
plt.show()

In [ ]:
result.show()

In [ ]:
import cv2
import torch
from ultralytics import YOLO

# Load YOLOv8 model (replace with your trained custom weights path)
model = YOLO("yolov8n.pt")  # Replace with the correct path to your trained model

# Open webcam
cap = cv2.VideoCapture(0)  # Use 0 for default webcam

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Failed to grab frame")
        break

    # Perform detection
    results = model(frame)  # Get predictions

    # Render the results on the frame
    rendered_img = results[0].plot()  # Get the processed frame with detections

    # Display the output
    cv2.imshow("Helmet & Number Plate Detection", rendered_img)

    # Quit on 'q' key
    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
import uuid
import os
import time

In [ ]:
IMAGES_PATH =os.path.join('data','images')
labels=('Helmet', 'Human head', 'Motorcycle', 'Vehicle registration plate')
number_images=5

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [ ]:
!cd yolov5 && python train.py --img 320 --batch 16 --epochs 500 --data dataset.yaml --weights yolov5s.pt --workers 2

In [ ]:
#Custom model
model = torch.hub.load('ultralytics/yolov5', 'custom', path='last.pt', force_reload=True)

In [ ]:
img = os.path.join('data', 'images', '4a11e2446df66145.jpg')

In [ ]:
img = 'https://cdn.dnaindia.com/sites/default/files/styles/full/public/2019/09/14/867723-traffic-violations-091419.jpg'

In [ ]:
results = model(img)

In [ ]:
results

In [ ]:
%matplotlib inline
plt.imshow(np.squeeze(results.render()))
plt.show()

In [ ]:
cap = cv2.VideoCapture(0)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Make detections
    results = model(frame)

    cv2.imshow('YOLO', np.squeeze(results.render()))

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break
cap.release()
cv2.destroyAllWindows()

In [ ]:
!pip install pytesseract


In [ ]:
import pytesseract
import urllib.request

In [ ]:
import os
import shutil

tesseract_cmd = os.environ.get("TESSERACT_CMD") or shutil.which("tesseract")
if not tesseract_cmd and os.name == "nt":
    for candidate in (
        os.path.join(os.environ.get("ProgramFiles", ""), "Tesseract-OCR", "tesseract.exe"),
        os.path.join(os.environ.get("ProgramFiles(x86)", ""), "Tesseract-OCR", "tesseract.exe"),
    ):
        if candidate and os.path.exists(candidate):
            tesseract_cmd = candidate
            break

if tesseract_cmd:
    pytesseract.pytesseract.tesseract_cmd = tesseract_cmd
else:
    raise FileNotFoundError(
        "Tesseract OCR binary not found. Install tesseract-ocr or set TESSERACT_CMD."
    )

In [ ]:
IMAGES_PATH = os.path.join('data', 'images')
labels = ('Helmet', 'Human head', 'Motorcycle', 'Vehicle registration plate')
number_images = 5

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [ ]:
img_url = 'https://img.thedailybeast.com/image/upload/c_crop,d_placeholder_euli9k,h_1440,w_2560,x_0,y_0/dpr_1.5/c_limit,w_1044/fl_lossy,q_auto/v1499473766/170707-Neidermeyer-india-car-tease_z6ld4i'

results = model(img_url)
boxes = results.xyxy[0].cpu().numpy()

with urllib.request.urlopen(img_url) as url:
    s = url.read()
arr = np.asarray(bytearray(s), dtype=np.uint8)
img_data = cv2.imdecode(arr, -1)

for box in boxes:
    if labels[int(box[5])] == 'Vehicle registration plate':
        region = img_data[int(box[1]):int(box[3]), int(box[0]):int(box[2])]
        if region.size == 0 or region.shape[0] == 0:
            continue

        ratio = region.shape[1] / region.shape[0]
        model_height = 32
        resized_plate = cv2.resize(
            region, (max(1, int(model_height * ratio)), model_height), interpolation=cv2.INTER_LINEAR
        )

        ocr_result = pytesseract.image_to_string(resized_plate)
        print(ocr_result)
        plt.imshow(cv2.cvtColor(region, cv2.COLOR_BGR2RGB))
        cv2.putText(
            img_data, ocr_result, (int(box[0]), int(box[1])), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2, cv2.LINE_AA
        )

plt.imshow(cv2.cvtColor(img_data, cv2.COLOR_BGR2RGB))
plt.show()

In [ ]:
if len(boxes) > 0:
    scores = boxes[:, 4]
    best_index = np.argmax(scores)

    for i, box in enumerate(boxes):
        if i == best_index and labels[int(box[5])] == 'Vehicle registration plate':
            margin = 10
            box = [
                max(0, box[0] - margin),
                max(0, box[1] - margin),
                min(img_data.shape[1], box[2] + margin),
                min(img_data.shape[0], box[3] + margin),
            ]

            region = img_data[int(box[1]):int(box[3]), int(box[0]):int(box[2])]
            if region.size == 0 or region.shape[0] == 0:
                continue

            gray = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
            _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

            ratio = thresh.shape[1] / thresh.shape[0]
            model_height = 32
            resized_plate = cv2.resize(
                thresh, (max(1, int(model_height * ratio)), model_height), interpolation=cv2.INTER_LINEAR
            )

            custom_config = r'--oem 3 --psm 6'
            ocr_result = pytesseract.image_to_string(resized_plate, config=custom_config)
            print(ocr_result)

            plt.imshow(cv2.cvtColor(region, cv2.COLOR_BGR2RGB))
            plt.show()

In [ ]:
import glob
import pandas as pd

results_files = sorted(glob.glob("runs/**/results.csv", recursive=True))
if not results_files:
    raise FileNotFoundError("No training results.csv found under 'runs/'. Run training first.")

df = pd.read_csv(results_files[-1])
# Remove leading and trailing spaces from column names
df.columns = df.columns.str.strip()

map50_col = "metrics/mAP_0.5" if "metrics/mAP_0.5" in df.columns else "metrics/mAP50(B)"
map5095_col = "metrics/mAP_0.5:0.95" if "metrics/mAP_0.5:0.95" in df.columns else "metrics/mAP50-95(B)"

average_mAP_05 = df[map50_col].mean()
average_mAP_05095 = df[map5095_col].mean()

print(f"Loaded results from: {results_files[-1]}")
print(f"Average mAP@0.5: {average_mAP_05 * 100:.2f}%")
print(f"Average mAP@0.5:0.95: {average_mAP_05095 * 100:.2f}%")